# PCL Binary Classification with RoBERTa

Binary classification of Patronizing and Condescending Language (PCL):
- **0**: non-PCL
- **1**: contains PCL

**Data sources:**
- `dontpatronizeme_pcl.tsv` — full corpus with text
- `train_semeval_parids-labels.csv` — train split (7-annotator label vectors)
- `dev_semeval_parids-labels.csv` — validation split (7-annotator label vectors)

**Label binarisation (SemEval CSVs):** all-zeros → 0 (non-PCL); any non-zero → 1 (PCL)

## Imports & Configuration

In [61]:
import re
import ast
import string
from pathlib import Path

import pandas as pd
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    RobertaTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)
from tqdm.notebook import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR = Path(os.getcwd())
DATA_DIR = (BASE_DIR / 'data').resolve()
TSV_PATH   = DATA_DIR / 'dontpatronizeme_pcl.tsv'
TRAIN_CSV  = DATA_DIR / 'train_semeval_parids-labels.csv'
DEV_CSV    = DATA_DIR / 'dev_semeval_parids-labels.csv'
BEST_MODEL_DIR = BASE_DIR / 'BestModel'
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = BEST_MODEL_DIR / 'best_roberta_pcl.pt'

# ── Hyperparameters ───────────────────────────────────────────────────────────
MODEL_NAME   = 'roberta-base'
BATCH_SIZE   = 16
MAX_LEN      = 256
EPOCHS       = 20
LR           = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1        
SEED         = 42
PATIENCE     = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')

Device : cuda
Model  : roberta-base


## Text Preprocessing

In [62]:
_PUNCT_TABLE = str.maketrans('', '', string.punctuation)

def clean_text(text: str) -> str:
    """Clean text by removing HTML tags and punctuation."""
    text = re.sub(r'<[^>]+>', ' ', text)          # strip HTML tags
    text = text.translate(_PUNCT_TABLE)            # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    return text

## Loading Datasets

In [47]:
def load_pcl_splits(data_dir):
    """Load train and dev maps and TSV data."""
    # Load train and dev maps
    train_map = pd.read_csv(data_dir / "train_semeval_parids-labels.csv")
    dev_map = pd.read_csv(data_dir / "dev_semeval_parids-labels.csv")
    
    # Load TSV (The actual text)
    tsv = pd.read_csv(
        data_dir / "dontpatronizeme_pcl.tsv",
        sep="\t",
        skiprows=4,
        names=["id", "art_id", "keyword", "country", "text", "label_tsv"],
        on_bad_lines='skip'
    )

    # 1. Ensure IDs are integers for the merge
    tsv = tsv.dropna(subset=['id', 'text'])
    tsv['id'] = tsv['id'].astype(int)
    train_map['par_id'] = train_map['par_id'].astype(int)
    dev_map['par_id'] = dev_map['par_id'].astype(int)

    # 2. Convert PCL label strings to binary labels
    def parse_binary(label_str):
        # Convert string representation of list to actual list
        lst = ast.literal_eval(label_str)
        return 1 if sum(lst) > 0 else 0

    train_map['binary_label'] = train_map['label'].apply(parse_binary)
    dev_map['binary_label'] = dev_map['label'].apply(parse_binary)

    # 3. Merge text from TSV onto the labels from CSV
    train = train_map.merge(tsv[['id', 'text']], left_on='par_id', right_on='id')
    dev = dev_map.merge(tsv[['id', 'text']], left_on='par_id', right_on='id')
    
    # Standardise and clean
    train = train[['text', 'binary_label']].rename(columns={'binary_label': 'label'})
    dev = dev[['text', 'binary_label']].rename(columns={'binary_label': 'label'})
    
    train['text'] = train['text'].astype(str).apply(clean_text)
    dev['text'] = dev['text'].astype(str).apply(clean_text)
    
    return train, dev

    def process_labels(df):
        # Convert string "[0,0,1...]" to actual list if necessary
        if isinstance(df['label'].iloc[0], str):
            df["label_vec"] = df["label"].apply(ast.literal_eval)
        else:
            df["label_vec"] = df["label"]
            
        # PCL Task 1: Binary (1 if any category is present, else 0)
        df["binary_label"] = df["label_vec"].apply(lambda v: 1 if sum(v) > 0 else 0)
        return df

    train_map = process_labels(train_map)
    dev_map = process_labels(dev_map)

    # Merge: par_id (from CSV) matches id (from TSV)
    train = train_map.merge(tsv[["id", "text"]], left_on="par_id", right_on="id")
    dev = dev_map.merge(tsv[["id", "text"]], left_on="par_id", right_on="id")
    
    # Cleanup and Clean Text
    train = train[["text", "binary_label"]].rename(columns={"binary_label": "label"})
    dev = dev[["text", "binary_label"]].rename(columns={"binary_label": "label"})
    
    train["text"] = train["text"].astype(str).apply(clean_text)
    dev["text"] = dev["text"].astype(str).apply(clean_text)
    
    return train, dev

train_df, dev_df = load_pcl_splits(DATA_DIR)
print(f"Train: {len(train_df)} samples, Dev: {len(dev_df)} samples")

Train: 8375 samples, Dev: 2093 samples


In [48]:
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.texts    = df['text'].astype(str).tolist()
        self.labels   = df['label'].tolist() 
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        text = self.texts[idx]
        label = self.labels[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )

        return {
            'input_ids':      enc['input_ids'].squeeze(0),     
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(label, dtype=torch.long),
        }

# Re-initialise with the standardised dataframes
train_ds = PCLDataset(train_df, tokenizer, MAX_LEN)
dev_ds   = PCLDataset(dev_df,   tokenizer, MAX_LEN)

print(f'Train dataset size : {len(train_ds):,}')
print(f'Dev   dataset size : {len(dev_ds):,}')

Train dataset size : 8,375
Dev   dataset size : 2,093


In [63]:
# 1. Extract labels from the processed train_df
train_labels = np.array(train_df['label'].tolist())

# 2. Calculate class distribution and weights
class_counts = np.bincount(train_labels)  # [count_0, count_1]
# Use inverse frequency: higher weight for the minority class (PCL)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

# 3. Create a weighted sampler
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True,
)

# 4. Initialise DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
    pin_memory=(DEVICE.type == 'cuda')
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == 'cuda'),
)

print(f'Class counts (train)  : non-PCL={class_counts[0]:,} | PCL={class_counts[1]:,}')
print(f'Class weights         : non-PCL={class_weights[0]:.6f} | PCL={class_weights[1]:.6f}')
print(f'Train batches/epoch   : {len(train_loader):,}')
print(f'Dev   batches/epoch   : {len(dev_loader):,}')

Class counts (train)  : non-PCL=7,581 | PCL=794
Class weights         : non-PCL=0.000132 | PCL=0.001259
Train batches/epoch   : 524
Dev   batches/epoch   : 131


## Model, Optimiser & Scheduler

In [65]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Total training steps : {total_steps:,}')
print(f'Warmup steps         : {warmup_steps:,}')
print(f'Model parameters     : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total training steps : 10,480
Warmup steps         : 1,048
Model parameters     : 124,647,170


## Training Loop

In [66]:
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    """Evaluate model on dev set."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    acc      = accuracy_score(all_labels, all_preds)
    f1_bin   = f1_score(all_labels, all_preds, average='binary', pos_label=1, zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro',  zero_division=0)
    report   = classification_report(
        all_labels, all_preds,
        target_names=['non-PCL', 'PCL'],
        zero_division=0,
    )
    return {'accuracy': acc, 'f1_binary': f1_bin, 'f1_macro': f1_macro, 'report': report}


best_f1 = 0.0
epochs_no_improve = 0
history = []

total_steps = len(train_loader) * EPOCHS
pbar = tqdm(total=total_steps, desc='Training', leave=True, dynamic_ncols=True)

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    running_loss = 0.0
    pbar.reset(total=len(train_loader))
    pbar.set_description(f'Epoch {epoch}/{EPOCHS}')

    for step, batch in enumerate(train_loader, start=1):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        running_loss += loss.item()
        avg_loss = running_loss / step
        
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        pbar.set_postfix(loss=f'{loss.item():.4f}', avg=f'{avg_loss:.4f}')
        pbar.update(1)


    # Validation
    metrics = evaluate(model, dev_loader)

    train_avg_loss = running_loss / len(train_loader)

    # Save history using the final average loss of the epoch
    history.append({
        'epoch':      epoch,
        'train_loss': train_avg_loss,
        'accuracy':   metrics['accuracy'],
        'f1_binary':  metrics['f1_binary'],
        'f1_macro':   metrics['f1_macro'],
    })

    # Clearer formatting for the console
    print(f'Epoch {epoch}/{EPOCHS} | Loss: {train_avg_loss:.4f} | Acc: {metrics["accuracy"]:.4f} | F1: {metrics["f1_binary"]:.4f} | F1 Macro: {metrics["f1_macro"]:.4f}')    
    
    if metrics['f1_binary'] > best_f1:
        best_f1 = metrics['f1_binary']
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'  New Best F1: {best_f1:.4f} (Checkpoint Saved)')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping triggered at epoch {epoch}')
            break

pbar.close()

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
print(f'Loaded best checkpoint (F1: {best_f1:.4f})')

# Final Summary
print('\n' + '='*30 + '\n TRAINING COMPLETE \n' + '='*30)
hist_df = pd.DataFrame(history)
print(hist_df.to_string(index=False))

Training:   0%|          | 0/10480 [00:00<?, ?it/s]

Epoch 1/20 | Loss: 0.5513 | Acc: 0.7926 | F1: 0.4407 | F1 Macro: 0.6567
  New Best F1: 0.4407 (Checkpoint Saved)
Epoch 2/20 | Loss: 0.2714 | Acc: 0.8333 | F1: 0.4830 | F1 Macro: 0.6918
  New Best F1: 0.4830 (Checkpoint Saved)
Epoch 3/20 | Loss: 0.1845 | Acc: 0.9216 | F1: 0.5859 | F1 Macro: 0.7713
  New Best F1: 0.5859 (Checkpoint Saved)
Epoch 4/20 | Loss: 0.1251 | Acc: 0.9040 | F1: 0.5856 | F1 Macro: 0.7656
Epoch 5/20 | Loss: 0.0904 | Acc: 0.9145 | F1: 0.5866 | F1 Macro: 0.7695
  New Best F1: 0.5866 (Checkpoint Saved)
Epoch 6/20 | Loss: 0.0507 | Acc: 0.9298 | F1: 0.6370 | F1 Macro: 0.7991
  New Best F1: 0.6370 (Checkpoint Saved)
Epoch 7/20 | Loss: 0.0372 | Acc: 0.9016 | F1: 0.5913 | F1 Macro: 0.7677
Epoch 8/20 | Loss: 0.0304 | Acc: 0.9212 | F1: 0.6207 | F1 Macro: 0.7884
Early stopping triggered at epoch 8
Loaded best checkpoint (F1: 0.6370)

 TRAINING COMPLETE 
 epoch  train_loss  accuracy  f1_binary  f1_macro
     1    0.551270  0.792642   0.440722  0.656724
     2    0.271399  0.8332

## Generate Dev and Test Predictions

In [67]:
def predict(model: nn.Module, loader: DataLoader) -> list[int]:
    """Return predictions (0 or 1) in loader order."""
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Predicting', leave=False):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            preds = model(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().tolist())
    return all_preds


# Dev predictions
dev_preds = predict(model, dev_loader)
Path('dev.txt').write_text('\n'.join(map(str, dev_preds)) + '\n')
print(f'dev.txt written  — {len(dev_preds)} lines')


# Test set
test_df = pd.read_csv(
    DATA_DIR / 'task4_test.tsv',
    sep='\t',
    header=None,
    
    names=['idx', 'par_id', 'keyword', 'country', 'text'],
)
test_df['text'] = test_df['text'].apply(clean_text)
test_df.reset_index(drop=True, inplace=True)

test_ds = PCLDataset(
    test_df.assign(label=0),
    tokenizer,
    MAX_LEN,
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_preds = predict(model, test_loader)
Path('test.txt').write_text('\n'.join(map(str, test_preds)) + '\n')
print(f'test.txt written — {len(test_preds)} lines')

Predicting:   0%|          | 0/131 [00:00<?, ?it/s]

dev.txt written  — 2093 lines


Predicting:   0%|          | 0/240 [00:00<?, ?it/s]

test.txt written — 3832 lines
